# 1kg_eur — UKB-style PCA (rare + common), Vaughan-Williams-style parameters

Replicates, as closely as AoU allows, the PC construction from
**"Evaluating confounding in rare variant genome-wide association studies"**
(Nat Commun 2026, `s41467-026-73776-9`) on our CEU/GBR-anchored round-2 cohort,
so we can see whether their rare-variant confounding signal reproduces here.

Their parameters, and what we do instead where we cannot match:

| | Paper (UKB) | Here (AoU) |
|---|---|---|
| Data | WES, 306,991 unrelated EUR | WGS ACAF panel, round-2 `1kg_eur` keep list |
| Common PCs | 147,604 genotyped autosomal variants "in linkage equilibrium", **projected** onto a pre-derived UKB reference | same MAF and r²=0.1, pruned in-sample; **no external reference exists for AoU**, so fit in-sample |
| Rare PCs | 617,375 WES SNPs, MAF < 1%, MAC > 50, r² > 0.1 in 10 Mb / 2 Mb-step windows | same MAF ceiling and r²/window; **MAC floor is not reachable** — see below |
| Depth QC | `90pct10dp` (≥90% of genotypes at depth ≥10) | no per-variant depth in the ACAF pgen; nearest proxy is `--geno 0.10` |
| HWE | not applied | not applied (matches; and matches our own rare-variant practice) |
| Long-range LD / MHC | not excluded | **not excluded** — deliberately, to stay faithful. Our production nb03 does exclude them |
| Odd/even split | PCs refit independently on odd and even chromosomes | same |
| Software | `flashpcaR` | `plink2 --pca approx` |

**The MAC floor is the one parameter we genuinely cannot match.** They allow
MAC > 50, which in 306,991 exomes is MAF ≈ 0.008%. Our unified panel was built
with a pooled `--maf 0.001` floor, so nothing below 0.1% exists in it — and at
n ≈ 222K, MAF 0.1% is MAC ≈ 445. Our "rare" band is therefore **0.1% ≤ MAF < 1%**,
an order of magnitude less rare than theirs. If the odd/even diagnostic below
comes back clean, that is the first thing to blame, not evidence of no confounding.

Their 147,604 / 617,375 are outcomes of their QC on their data, not parameters —
we match the filters and let the counts land where they land. Ours will be larger
(WGS, whole genome, not exome-restricted); the counts are reported for context.

**Runs:** one Batch job does panel download, QC, pruning and all six PCA fits
(2 arms × {all, odd, even}); only eigenvectors come back to the VM. Plots local.


## Config

In [ ]:
import os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAMPLE_SET      = "1kg_eur"          # CEU/GBR-anchored round-2 cohort
CDR_VERSION     = "v9"
PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/{SAMPLE_SET}"

# input: raw unified panel (pooled MAF floor 0.001) — the rare band only exists here,
# not in r1_qc/r2_qc, which are already MAF 0.01 filtered
PANEL_GS   = f"{WS_GS}/01_ancestry_filtering/unified_panel/unified_panel_{CDR_VERSION}"
KEEP_GS    = f"{R_GS}/01_ancestry/round2/1kg_eur_keep_ids.txt"   # pre-CEUGBR-rename filename
PLINK2_GS  = f"{R_GS}/03_grm/bin/plink2"                          # staged by 04_grm_panel_qc
COVPCA_GS  = f"{R_GS}/01_ancestry/covariate_pca/covariate_pcs_1kg_CEUGBR.txt"  # nb03 output

OUT_GS = f"{R_GS}/01_ancestry/ukb_style_pca"
LOG_GS = f"{R_GS}/dsub_logs"
LOCAL  = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_ukbpca")
os.makedirs(LOCAL, exist_ok=True)

# ── UKB-style parameters ──────────────────────────────────────────────────────
COMMON_MAF     = 0.01        # >1% MAF
COMMON_PRUNE   = "1000kb 0.1"

RARE_MAF_MAX   = 0.01        # <1% MAF
RARE_MAC_MIN   = 51          # their ">50" — unreachable below the panel's 0.1% floor
RARE_PRUNE     = "10000kb 0.1"   # 10 Mb window. plink2 steps by 1 variant, not 2 Mb —
                                 # strictly more aggressive than the paper, never less

GENO_MAX       = 0.10        # proxy for their 90pct10dp depth filter
N_PCS_FIT      = 20

JOB_MACHINE, JOB_DISK_GB = "n1-highmem-32", 600

print(f"panel:  {PANEL_GS}")
print(f"keep:   {KEEP_GS}")
print(f"output: {OUT_GS}")
print(f"local:  {LOCAL}")

## Install dsub

In [ ]:
subprocess.run(["bash", "-c", f"""
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"
find "$DSUB_DIR" -name '*.pyc' -delete
echo "dsub: $(dsub --version)"
"""], check=True)

## Check inputs exist

The plink2 binary is the one `04_grm_panel_qc.ipynb` staged. If it is missing,
run that notebook's staging cell first rather than staging a different build —
version skew between arms would be a silent confounder of its own.

In [ ]:
missing = subprocess.run(["bash", "-c", f"""
for P in "{PANEL_GS}.pgen" "{PANEL_GS}.pvar" "{PANEL_GS}.psam" "{KEEP_GS}" "{PLINK2_GS}"; do
  gcloud storage ls "$P" >/dev/null 2>&1 || echo "$P"
done
"""], capture_output=True, text=True).stdout.split()

for p in missing:
    print(f"MISSING: {p}")
if not missing:
    print("all inputs present")

# plink2 is the one we can fix here; anything else means an upstream notebook has not run
if PLINK2_GS in missing:
    print("\n-> run the staging cell below, then re-run this check")
assert not [p for p in missing if p != PLINK2_GS], \
    "missing inputs that this notebook cannot create - run the upstream notebook first"

# nb03 comparison is optional, so this one only warns
subprocess.run(["bash", "-c",
    f'gcloud storage ls "{COVPCA_GS}" >/dev/null 2>&1'
    f' && echo "ok: nb03 covariate PCs" || echo "note: {COVPCA_GS} absent - nb03 comparison will be skipped"'],
    check=False)

## Stage plink2

`04_grm_panel_qc.ipynb` / `06_grm_shards.ipynb` stage this binary, but they may not
have run in this workspace. Safe to re-run: skips if already present. Uses the VM's
own `~/bin/plink2` so the Batch worker runs the same build the local cells do.

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
if gcloud storage ls "{PLINK2_GS}" >/dev/null 2>&1; then
  echo "already staged: {PLINK2_GS}"
elif [ -x "$HOME/bin/plink2" ]; then
  gcloud storage cp "$HOME/bin/plink2" "{PLINK2_GS}" && echo "staged: {PLINK2_GS}"
else
  echo "no local plink2 - installing to \\$HOME/bin first"
  BIN_DIR="$HOME/bin"; mkdir -p "$BIN_DIR"
  cd /tmp
  wget -q -O plink2.zip "https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR" && chmod +x "$BIN_DIR/plink2"
  gcloud storage cp "$BIN_DIR/plink2" "{PLINK2_GS}" && echo "staged: {PLINK2_GS}"
fi
"$HOME/bin/plink2" --version 2>/dev/null || true
"""], check=True)

## The Batch job

One task, six PCA fits. Per arm:

1. `--keep` the round-2 cohort, then QC — MAF band, MAC floor, `--geno`, biallelic
   only. No HWE, no long-range-LD exclusion: both match the paper.
2. LD-prune at r² = 0.1 (1 Mb common, 10 Mb rare).
3. `--pca approx 20 allele-wts` on all autosomes, then again on odd and on even
   chromosomes separately.

Only `.eigenvec`, `.eigenval`, `.eigenvec.allele` and the variant-count log come
back — the intermediate pgens stay on the worker's disk and die with it.

Note `--nonfounders`: the round-2 keep list still contains relatives (the paper
excludes everyone related to third degree or closer). That difference is examined
in the diagnostics, not fixed here — fixing it would mean a different sample and
break comparability with everything else in this pipeline.

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"
find "$DSUB_DIR" -name '*.pyc' -delete

dsub \
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \
  --logging "{LOG_GS}" \
  --service-account "{SERVICE_ACCOUNT}" \
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "ukb-style-pca" \
  --machine-type "{JOB_MACHINE}" --disk-size "{JOB_DISK_GB}" \
  --input  PLINK2="{PLINK2_GS}" \
  --input  KEEP="{KEEP_GS}" \
  --env    PANEL_GS="{PANEL_GS}" \
  --env    COMMON_MAF="{COMMON_MAF}" \
  --env    COMMON_PRUNE="{COMMON_PRUNE}" \
  --env    RARE_MAF_MAX="{RARE_MAF_MAX}" \
  --env    RARE_MAC_MIN="{RARE_MAC_MIN}" \
  --env    RARE_PRUNE="{RARE_PRUNE}" \
  --env    GENO_MAX="{GENO_MAX}" \
  --env    N_PCS="{N_PCS_FIT}" \
  --output-recursive OUT="{OUT_GS}" \
  --command '
    set -eo pipefail
    chmod +x "$PLINK2"
    NT=$(nproc)
    mkdir -p /mnt/data/panel /mnt/data/work "$OUT"
    cd /mnt/data/work

    echo "=== downloading unified panel"
    gcloud storage cp "$PANEL_GS".pgen "$PANEL_GS".pvar "$PANEL_GS".psam /mnt/data/panel/
    PANEL=/mnt/data/panel/$(basename "$PANEL_GS")

    for ARM in common rare; do
      echo "=== arm: $ARM — QC"
      if [ "$ARM" = common ]; then
        FREQ_FLAGS="--maf $COMMON_MAF"
        PRUNE_PARAMS="$COMMON_PRUNE"
      else
        FREQ_FLAGS="--max-maf $RARE_MAF_MAX --mac $RARE_MAC_MIN"
        PRUNE_PARAMS="$RARE_PRUNE"
      fi

      "$PLINK2" --pfile "$PANEL" --keep "$KEEP" --nonfounders \
        $FREQ_FLAGS --geno "$GENO_MAX" --max-alleles 2 \
        --threads $NT --make-pgen --out "$ARM"_qc

      echo "=== arm: $ARM — LD prune at $PRUNE_PARAMS"
      "$PLINK2" --pfile "$ARM"_qc --nonfounders \
        --indep-pairwise $PRUNE_PARAMS \
        --threads $NT --out "$ARM"_prune

      echo "$ARM: $(grep -vc "^##" "$ARM"_qc.pvar) after QC, $(wc -l < "$ARM"_prune.prune.in) after pruning" \
        | tee -a "$OUT"/variant_counts.txt

      "$PLINK2" --pfile "$ARM"_qc --nonfounders \
        --extract "$ARM"_prune.prune.in \
        --threads $NT --make-pgen --out "$ARM"_final

      for SPLIT in all odd even; do
        case $SPLIT in
          all)  CHRFLAG="" ;;
          odd)  CHRFLAG="--chr 1,3,5,7,9,11,13,15,17,19,21" ;;
          even) CHRFLAG="--chr 2,4,6,8,10,12,14,16,18,20,22" ;;
        esac
        echo "=== arm: $ARM split: $SPLIT — PCA"
        "$PLINK2" --pfile "$ARM"_final --nonfounders $CHRFLAG \
          --freq counts --pca approx "$N_PCS" allele-wts \
          --threads $NT --out "$ARM"_"$SPLIT"
        cp "$ARM"_"$SPLIT".eigenvec "$ARM"_"$SPLIT".eigenval "$OUT"/
        cp "$ARM"_"$SPLIT".eigenvec.allele "$OUT"/ || true
      done
    done
    echo "=== done"
    cat "$OUT"/variant_counts.txt
  '
"""], check=True)

## Monitor

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "{PROJECT_ID}" --location "{REGION}" \
  --jobs "ukb-style-pca*" --status '*' --full 2>&1 | tail -40
"""], check=False)

## Download eigenvectors

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{OUT_GS}/*.eigenvec" "{OUT_GS}/*.eigenval" "{LOCAL}/"
gcloud storage cp "{OUT_GS}/variant_counts.txt" "{LOCAL}/" || true
ls -la "{LOCAL}" | tail -20
"""], check=True)

print()
print(open(f"{LOCAL}/variant_counts.txt").read()
      if os.path.isfile(f"{LOCAL}/variant_counts.txt") else "variant_counts.txt not found")

## Load

`--pca approx` writes in-sample eigenvectors directly, so unlike nb03 there is no
`--score` step — nothing is being projected onto an external reference here
(which is itself a departure from the paper's common-variant arm).

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS_FIT + 1)]

def read_eigenvec(arm, split):
    d = pd.read_csv(f"{LOCAL}/{arm}_{split}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    return d[["person_id"] + PC_COLS]

def read_eigenval(arm, split):
    ev = np.loadtxt(f"{LOCAL}/{arm}_{split}.eigenval")
    return ev / ev.sum() * 100

pcs = {(a, s): read_eigenvec(a, s)
       for a in ("common", "rare") for s in ("all", "odd", "even")}
pct = {(a, s): read_eigenval(a, s)
       for a in ("common", "rare") for s in ("all", "odd", "even")}

n = len(pcs[("rare", "all")])
print(f"{n:,} participants")
for arm in ("common", "rare"):
    print(f"{arm:7s} " + "  ".join(f"PC{k+1} {pct[(arm,'all')][k]:.2f}%" for k in range(5)))

## Scree — common vs rare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, arm in zip(axes, ("common", "rare")):
    ax.bar(range(1, N_PCS_FIT + 1), pct[(arm, "all")])
    ax.set_xlabel("PC"); ax.set_ylabel("% variance explained"); ax.set_title(arm)
plt.suptitle(f"1kg_eur UKB-style PCA — scree (n={n:,})")
plt.tight_layout()
plt.savefig(f"{LOCAL}/ukbpca_scree.png", dpi=150)
plt.show()

## PC pairs

In [ ]:
rng = np.random.default_rng(42)
idx = rng.choice(n, size=min(30_000, n), replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, arm in enumerate(("common", "rare")):
    d = pcs[(arm, "all")]
    for col, (i, j) in enumerate([(1, 2), (3, 4), (5, 6)]):
        ax = axes[row, col]
        ax.scatter(d[f"PC{i}"].iloc[idx], d[f"PC{j}"].iloc[idx],
                   s=1, alpha=0.1, color="0.3", rasterized=True)
        ax.set_xlabel(f"PC{i}"); ax.set_ylabel(f"PC{j}")
        ax.set_title(f"{arm}: PC{i} vs PC{j}")
plt.suptitle("1kg_eur UKB-style PCA — common (top) vs rare (bottom)")
plt.tight_layout()
plt.savefig(f"{LOCAL}/ukbpca_pc_pairs.png", dpi=150)
plt.show()

## Odd/even replication — the diagnostic worth having

The paper's sharpest trick. If a PC is tracking real, genome-wide population
structure, the odd-chromosome fit and the even-chromosome fit should recover
essentially the same axis, so |r| between them is high. If a PC is tracking one
locus, an unexcluded long-range-LD region, or assay noise, it lives on one set of
chromosomes and the correlation collapses.

Note the PCs are only identified up to sign and, where eigenvalues are close, up
to rotation within a subspace — so we report `max |r|` of each odd PC against *all*
even PCs, not just the diagonal.

In [ ]:
def odd_even_table(arm):
    o = pcs[(arm, "odd")].set_index("person_id")[PC_COLS]
    e = pcs[(arm, "even")].set_index("person_id")[PC_COLS]
    common_ids = o.index.intersection(e.index)
    o, e = o.loc[common_ids], e.loc[common_ids]
    R = np.corrcoef(o.to_numpy().T, e.to_numpy().T)[:N_PCS_FIT, N_PCS_FIT:]
    return pd.DataFrame({
        "PC":          range(1, N_PCS_FIT + 1),
        "diag_abs_r":  np.abs(np.diag(R)),
        "max_abs_r":   np.abs(R).max(axis=1),
        "best_match":  np.abs(R).argmax(axis=1) + 1,
        "pct_var":     pct[(arm, "all")][:N_PCS_FIT],
    }).set_index("PC")

oe = {arm: odd_even_table(arm) for arm in ("common", "rare")}
for arm in ("common", "rare"):
    print(f"\n── {arm} ──")
    print(oe[arm].round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for arm, style in (("common", "-o"), ("rare", "-s")):
    ax.plot(oe[arm].index, oe[arm]["max_abs_r"], style, label=arm, ms=4)
ax.axhline(0.9, color="0.7", ls="--", lw=1)
ax.set_xlabel("PC"); ax.set_ylabel("max |r| (odd vs even chromosomes)")
ax.set_ylim(0, 1.02); ax.set_xticks(range(1, N_PCS_FIT + 1)); ax.legend()
ax.set_title("Odd/even replication — PCs below the line are not genome-wide structure")
plt.tight_layout()
plt.savefig(f"{LOCAL}/ukbpca_odd_even.png", dpi=150)
plt.show()

## Loading spectra — is a PC driven by one region?

Companion to the odd/even test. That one asks whether a PC replicates across
chromosome sets; this one shows *where* along the genome its weight sits.

A PC tracking genuine population structure draws weight diffusely from the whole
genome. A PC tracking a single locus — an unexcluded long-range-LD region, an
assay artefact, a segregating inversion — shows as a spike over one region. The
paper excludes no long-range-LD regions in either arm, so this is worth looking at
directly rather than inferring it from the odd/even correlation alone.

`.eigenvec.allele` is one row per variant, so these files are large; only the
`all` split is fetched, and points are thinned for plotting.

In [ ]:
def load_allele_weights(arm, split="all"):
    """Fetch and read plink2's per-variant PC loadings for one arm."""
    path = f"{LOCAL}/{arm}_{split}.eigenvec.allele"
    if not os.path.isfile(path):
        subprocess.run(["gcloud", "storage", "cp",
                        f"{OUT_GS}/{arm}_{split}.eigenvec.allele", path], check=True)
    d = pd.read_csv(path, sep=r"\s+")          # plink2 output is whitespace-aligned
    d = d.rename(columns={"#CHROM": "CHROM"})
    d["CHROM"] = (d["CHROM"].astype(str).str.replace("chr", "", regex=False)
                  .astype(int))
    return d.sort_values(["CHROM", "POS"], ignore_index=True)


def plot_loadings(pc=1, arms=("common", "rare"), thin=40_000):
    fig, axes = plt.subplots(len(arms), 1, figsize=(14, 3.4 * len(arms)),
                             squeeze=False, sharex=True)
    for ax, arm in zip(axes[:, 0], arms):
        d = load_allele_weights(arm)
        col = f"PC{pc}"
        assert col in d.columns, f"{col} not in {d.columns.tolist()[:12]}"

        # cumulative x so chromosomes lay end to end
        sizes = d.groupby("CHROM")["POS"].max()
        offset = sizes.cumsum().shift(fill_value=0)
        x = d["POS"].to_numpy() + d["CHROM"].map(offset).to_numpy()

        keep = (np.linspace(0, len(d) - 1, thin).astype(int)
                if len(d) > thin else np.arange(len(d)))
        shade = d["CHROM"].to_numpy()[keep] % 2
        ax.scatter(x[keep], d[col].to_numpy()[keep], s=2, rasterized=True,
                   c=np.where(shade == 0, "0.25", "tab:blue"))
        ax.axhline(0, lw=0.8, color="0.6")
        ax.set_ylabel(f"{col} loading")
        ax.set_title(f"{arm} — {len(d):,} variants", loc="left", fontsize=10)

    axes[-1, 0].set_xticks(offset + sizes / 2)
    axes[-1, 0].set_xticklabels(sizes.index, fontsize=7)
    axes[-1, 0].set_xlabel("chromosome")
    plt.suptitle(f"UKB-style PCA — PC{pc} loadings along the genome")
    plt.tight_layout()
    plt.savefig(f"{LOCAL}/ukbpca_loadings_pc{pc}.png", dpi=150)
    plt.show()


plot_loadings(pc=1)

## Rare vs common — do the rare PCs carry structure the common PCs miss?

This is the paper's actual claim: rare-variant PCs capture fine-scale geography
that common-variant PCs do not, and that residual structure is what confounds
rare-variant association tests. If every rare PC is well predicted by the common
PCs (high R²), there is nothing extra here to confound anything.

Reported as the R² of each rare PC regressed on all 20 common PCs.

In [ ]:
cm = pcs[("common", "all")].set_index("person_id")[PC_COLS]
rr = pcs[("rare",   "all")].set_index("person_id")[PC_COLS]
ids = cm.index.intersection(rr.index)
X = np.column_stack([np.ones(len(ids)), cm.loc[ids].to_numpy()])
Y = rr.loc[ids].to_numpy()

beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
resid = Y - X @ beta
r2_on_common = 1 - resid.var(axis=0) / Y.var(axis=0)

tab = pd.DataFrame({
    "PC": range(1, N_PCS_FIT + 1),
    "rare_pct_var": pct[("rare", "all")][:N_PCS_FIT],
    "r2_on_common_pcs": r2_on_common,
    "odd_even_max_r": oe["rare"]["max_abs_r"].to_numpy(),
}).set_index("PC")
print(tab.round(3).to_string())
print("\nrare PCs that replicate odd/even (>0.9) but are poorly explained by common PCs (<0.5):")
print(list(tab.index[(tab["odd_even_max_r"] > 0.9) & (tab["r2_on_common_pcs"] < 0.5)]))

## Against our production covariate PCs

nb03's 20 PCs (HM3 common, r²=0.1, long-range LD excluded) are what actually go
into residualization. Same question as above, but against the PCs we really use:
anything the rare arm sees that these do not is structure our phenotypic
covariance estimates are currently not adjusting for.

In [ ]:
COV_LOCAL = f"{LOCAL}/covariate_pcs_1kg_CEUGBR.txt"
subprocess.run(["bash", "-c", f'gcloud storage cp "{COVPCA_GS}" "{COV_LOCAL}"'], check=False)

if os.path.isfile(COV_LOCAL):
    nb03 = pd.read_csv(COV_LOCAL, sep="\t")
    nb03["IID"] = nb03["IID"].astype(str)
    nb03 = nb03.set_index("IID")[PC_COLS]

    ids2 = nb03.index.intersection(rr.index)
    X2 = np.column_stack([np.ones(len(ids2)), nb03.loc[ids2].to_numpy()])
    for name, Z in (("rare", rr), ("common", cm)):
        Y2 = Z.loc[ids2].to_numpy()
        b2, *_ = np.linalg.lstsq(X2, Y2, rcond=None)
        r2 = 1 - (Y2 - X2 @ b2).var(axis=0) / Y2.var(axis=0)
        print(f"\nR² of UKB-style {name} PCs on nb03 covariate PCs (n={len(ids2):,}):")
        print("  " + "  ".join(f"PC{k+1} {r2[k]:.2f}" for k in range(10)))
else:
    print("nb03 covariate PCs not available — skipping")

## Write summary

In [ ]:
SUMMARY = f"{LOCAL}/ukb_style_pca_summary.tsv"
out = pd.concat(
    {arm: oe[arm].assign(arm=arm) for arm in ("common", "rare")}, names=["arm_", "PC"]
).reset_index(level=0, drop=True).reset_index()
out.loc[out["arm"] == "rare", "r2_on_common_pcs"] = r2_on_common
out.to_csv(SUMMARY, sep="\t", index=False, float_format="%.4f")
print(out.round(3).to_string(index=False))
print(f"\nwritten: {SUMMARY}")

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{SUMMARY}" "{OUT_GS}/ukb_style_pca_summary.tsv"
gcloud storage cp "{LOCAL}/ukbpca_"*.png "{OUT_GS}/figures/"
"""], check=True)
print(f"uploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/ukb_style_pca.ipynb')
_gs = f'{OUT_GS}/notebooks/ukb_style_pca.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')